In [1]:
!pip uninstall -y torch torchvision torchaudio -q
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128 -q
!pip install fair-esm -q # latest release, OR:
!pip install git+https://github.com/facebookresearch/esm.git  -q # bleeding edge, current repo main branch
!pip install biopython -q
!pip install torch_geometric -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.8.0+cu128.html -q
!sudo apt-get update -qq
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq dssp
!git clone https://github.com/Labrapuerta/graph_bind_uwu.git

import torch
from scipy.spatial import cKDTree
from Bio.PDB import PDBParser, Selection, PDBList, DSSP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch_geometric.data import Data
import esm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 18.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 74.8 MB/s eta 0:0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from torch_geometric.loader import DataLoader

# Ensure src is importable
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
!mkdir preprocessed
!mkdir data
!mkdir data/raw_pdbs

In [ ]:
# Paths
CSV_PATH = "pdb_splits.csv"
OUTPUT_DIR = "preprocessed"
RAW_PDB_DIR = "data/raw_pdbs"
ESM_CACHE_DIR = ".esm_cache"

# Processing settings
BATCH_SIZE = 2          # Proteins per batch for ESM (lower if OOM)
MAX_WORKERS = 8          # Parallel download workers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training settings
CV_FOLD = 1              # Validation fold (1-5)
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 32

print(f"Device: {DEVICE}")

In [ ]:
def convert_binding_format(binding_str: str) -> str:
    """Convert 'B_E103 B_R45' to 'E103 R45' format."""
    if pd.isna(binding_str) or not binding_str.strip():
        return ""

    tokens = binding_str.strip().split()
    converted = []

    for token in tokens:
        if "_" in token:
            parts = token.split("_", 1)
            if len(parts) == 2:
                converted.append(parts[1])
        else:
            converted.append(token)

    return " ".join(converted)


# Load and convert CSV
df_raw = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_raw)} rows from {CSV_PATH}")

print(f"\nBefore: {df_raw['Binding_Residues'].iloc[0][:50]}...")
df_raw["Binding_Residues"] = df_raw["Binding_Residues"].apply(convert_binding_format)
print(f"After:  {df_raw['Binding_Residues'].iloc[0][:50]}...")

# Save converted CSV
CONVERTED_CSV = "data/pdb_splits_converted.csv"
df_raw.to_csv(CONVERTED_CSV, index=False)
print(f"\nSaved to {CONVERTED_CSV}")

# Raw Data

## Splits to preprocess on different colabs

In [ ]:
split_1 = df_raw[:len(df_raw)//2]
split_1

In [ ]:
split_2 = df_raw[len(df_raw)//2:]
split_2

In [ ]:
# Data summary
print("Data Summary")
print("=" * 40)
print(f"Total proteins: {len(df_raw)}")
print(f"\nSplit distribution:")
print(split_1["Split"].value_counts())
print(f"\nCV Batch distribution (training):")
print(split_1[split_1["Split"] == "training"]["CV_Batch"].value_counts().sort_index())

## Save split to use it in the preprocessing
Replace by the split

In [ ]:
# Save converted CSV
split_CSV = "data/pdb_split_1_converted.csv"   ## Replace only the number in pdb_split_n
split_1.to_csv(split_CSV, index=False) ### Replace the dataframe in split_1
print(f"\nSaved to {split_CSV}")

# Preprocess everything

In [ ]:
from graph_bind_uwu.src.preprocessing.preprocess import preprocess

manifest_df = preprocess(
    csv_path=split_CSV,
    out_dir=OUTPUT_DIR,
    raw_dir=RAW_PDB_DIR,
    esm_cache=ESM_CACHE_DIR,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    max_workers=MAX_WORKERS,
)

In [ ]:
# Inspect manifest
print("Manifest Summary")
print("=" * 40)
print(f"Total: {len(manifest_df)}")
print(f"\nStatus:")
print(manifest_df["status"].value_counts())

ok_mask = manifest_df["status"].str.startswith("ok")
print(f"\nSample entries:")
manifest_df[ok_mask][["pdb_id", "chain", "split", "cv_batch", "n_residues", "n_binding"]].head()
safe = manifest_df[ok_mask][["pdb_id", "chain", "split", "cv_batch", "n_residues", "n_binding"]]


# Save converted CSV
manifest_CSV = "preprocessed/manifest.csv"
safe.to_csv(manifest_CSV, index=False)
print(f"\nSaved to {manifest_CSV}")

In [ ]:
!cp -r /content/preprocessed /content/drive/MyDrive/